# LectureSync — Colab engine
A free student initiative · CS Diploma, Faculty of Graduate Studies for Statistical Research (FGSSR), Cairo University · [cs-fgssr.pages.dev](https://cs-fgssr.pages.dev/)

**Steps:** press ▶ Run on the cell below → allow Google Drive → click the green **Connect to LectureSync** button. Keep this tab open while you transfer files.

> If Colab says *"This notebook was not authored by Google"*, press **Run anyway** — the full code is right here for you to read.

In [ ]:
#@title ▶ Start LectureSync engine
import os, re, secrets, subprocess, sys, threading, time, urllib.request
from google.colab import drive
from IPython.display import HTML, display

drive.mount('/content/drive')
print('Installing tools, about a minute the first time...')
subprocess.run('pip -q install telethon cryptg qrcode fastapi "uvicorn[standard]"', shell=True, check=True)
subprocess.run('apt-get -qq install -y p7zip-full p7zip-rar > /dev/null', shell=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

# Worker code comes from the same public repo this notebook lives in.
urllib.request.urlretrieve('https://raw.githubusercontent.com/92EPK/LectureSync/main/worker/server.py', '/content/server.py')

# Fresh random secret for this session only; it travels to the app inside the button link.
token = secrets.token_urlsafe(32)
os.environ['LS_ACCESS_TOKEN'] = token
subprocess.Popen([sys.executable, '-m', 'uvicorn', 'server:app', '--host', '127.0.0.1', '--port', '8000'],
                 cwd='/content', stdout=open('/content/worker.log', 'w'), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          stderr=subprocess.PIPE, text=True)
url = None
for line in tunnel.stderr:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stderr], daemon=True).start()

# Wait until the worker actually answers before showing the button.
for _ in range(60):
    try:
        urllib.request.urlopen(url + '/health', timeout=5); break
    except Exception as e:
        if getattr(e, 'code', None): break  # any HTTP answer means it's up
        time.sleep(2)

link = f'lecturesync://connect?url={url}&token={token}'
display(HTML(f"""
<div style="font-family:sans-serif;padding:16px;border:1px solid #2e7d32;border-radius:10px;max-width:560px">
  <p style="margin:0 0 12px">Engine is ready. Click the button, then choose <b>Open LectureSync</b> if your browser asks.</p>
  <a href="{link}" target="_blank" style="display:inline-block;background:#2e7d32;color:#fff;padding:10px 18px;border-radius:8px;text-decoration:none;font-weight:600">Connect to LectureSync</a>
  <p style="margin:12px 0 0;font-size:12px;color:#777">Button not working? In the app open “Connect with a link instead” and paste:<br><code>{url}</code></p>
</div>"""))
print('LECTURE_SYNC_URL=' + url)
while True:
    time.sleep(300)
    print(time.strftime('%H:%M'), 'worker alive')
